# 1. Data and OCR
Sources, recipes, Albanian local names, GBIF mapping, and OCR quality.

> Run after `python -m pipeline.rebuild` (README, Getting Started). The notebooks call the same modules as the scripts in `pipeline/` and `eval/`, so the results match the command line. Outputs never print Curious Cures transcription text, whose licence is not stated.

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from backend.app import db
db.init_db()
print("repo:", ROOT.name)

repo: EthnoLens-AI


## 1.1 What is in the database
Built by `python -m pipeline.rebuild`: MS Add. 9308 (Curious Cures, Cambridge) split into recipes, and Albanian names from Mustafa et al. (2012).

In [2]:
with db.connect() as conn:
    for table in ("documents", "passages", "records", "plant_names"):
        print(f"{table:12} {conn.execute(f'SELECT count(*) FROM {table}').fetchone()[0]:>5}")
    for row in conn.execute("SELECT id, source, language, period FROM documents WHERE collection != 'upload'"):
        print(dict(row))

documents        1
passages       100
records          0
plant_names    100
{'id': 'MS-ADD-09308', 'source': 'MS Add. 9308', 'language': 'Middle English, with some Latin', 'period': 'Late 14th/early 15th century'}


## 1.2 Recipes
`pipeline/load_cudl.py` splits the transcribed pages at recipe headings (*For…*, *To make…*, *Also…*). Only lengths are shown; the text itself is not redistributed.

In [3]:
import statistics
with db.connect() as conn:
    rows = conn.execute("SELECT id, folio, length(text) AS n FROM passages "
                        "WHERE document_id = 'MS-ADD-09308' ORDER BY seq").fetchall()
lengths = [r["n"] for r in rows]
print(f"{len(rows)} recipes, folio {rows[0]['folio']} to {rows[-1]['folio']}")
print(f"characters per recipe: median {statistics.median(lengths):.0f}, min {min(lengths)}, max {max(lengths)}")

100 recipes, folio 1v–2r to 30v–31r
characters per recipe: median 219, min 74, max 1323


## 1.3 Albanian local names (Mustafa et al. 2012, CC BY 2.0)
`pipeline/albanian_names.py` parsed Table 1 of the article into `data/processed/albanian_names.csv`; `pipeline/load_names.py` stores each name under its GBIF accepted name.

In [4]:
import csv
names = list(csv.DictReader(open("data/processed/albanian_names.csv", encoding="utf-8")))
print(len(names), "names for", len({n["canonical_name"] for n in names}), "species")
for n in names[:8]:
    print(f"{n['local_name']:22} {n['canonical_name']:28} {n['english_name']}")

100 names for 98 species
Bredhi i bardhë        Abies alba                   European silver fir
Hajdukati              Achillea millefolium         Yarrow
Pelini i egër (i zi)   Aconitum divergens           
Majdanozi i egër       Adiantum capillus-veneris    Southern maidenhair fern
Gështenja e egër       Aesculus hippocastanum       Horse chestnut
Pirrovina              Agropyron repens             Couch grass
Qepa                   Allium cepa                  Onion
Purrini                Allium porrum                Garden leek


## 1.4 Taxonomic mapping with GBIF
`pipeline/taxonomy.py`: *high* only for an exact species-level match, *medium* for spelling variants, *candidate* for genus-only, no match, or a near-miss that lands on a synonym of another species. Needs internet.

In [5]:
from pipeline.taxonomy import match
for name in ["Salvia officinalis", "Matricaria recutita", "Salvia oficinalis", "Thymus", "Artemisia absinthum"]:
    try:
        r = match(name)
        print(f"{name:22} -> {r['mapping_confidence']:9} {r['canonical_name']}  ({r['match_type']})")
    except Exception as error:
        print(f"{name:22} -> GBIF unreachable: {error}")

Salvia officinalis     -> high      Salvia officinalis  (EXACT)
Matricaria recutita    -> high      Matricaria chamomilla  (EXACT)
Salvia oficinalis      -> medium    Salvia officinalis  (VARIANT)


Thymus                 -> candidate None  (EXACT)
Artemisia absinthum    -> candidate None  (VARIANT)


## 1.5 OCR quality on printed pages
`python -m pipeline.ocr` runs Tesseract on 25 Biodiversity Heritage Library pages (FineBooks BHL IMPACT ground truth, CC BY 3.0) and stores the character error rate (CER) of each page.

In [6]:
import json, statistics
pages = [json.loads(line) for line in open("data/raw/bhl/birdsofgreatbrit02butl/pages.jsonl", encoding="utf-8")]
scored = [p for p in pages if "cer" in p]
print(f"{len(scored)} pages, mean CER {statistics.mean(p['cer'] for p in scored):.1%}, "
      f"median {statistics.median(p['cer'] for p in scored):.1%}")
for p in scored:
    print(f"{p['folio']:>7}  CER {p['cer']:.1%}")

25 pages, mean CER 2.7%, median 1.1%
   p. 1  CER 1.4%
   p. 2  CER 1.8%
   p. 3  CER 2.4%
   p. 6  CER 0.8%
   p. 7  CER 1.1%
  p. 10  CER 1.5%
  p. 11  CER 0.9%
  p. 12  CER 0.9%
  p. 13  CER 1.1%
  p. 16  CER 1.3%
  p. 17  CER 1.2%
  p. 18  CER 1.5%
  p. 19  CER 1.9%
  p. 20  CER 1.5%
  p. 21  CER 0.3%
  p. 24  CER 0.9%
  p. 25  CER 0.9%
  p. 28  CER 1.5%
  p. 29  CER 1.0%
  p. 30  CER 1.0%
  p. 31  CER 1.1%
  p. 34  CER 0.4%
  p. 37  CER 0.9%
  p. 38  CER 1.6%
  p. 39  CER 38.4%
